In [6]:
import numpy as np
from matplotlib import cm
import matplotlib.pyplot as plt
import glob
from scipy.optimize import fsolve
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.interpolate import splev
from scipy.interpolate import splrep
import colorsys
import networkx as nx
import random
import math
import matplotlib.style
import matplotlib as mpl
import pandas as pd
import graph_tool.all as gt
import seaborn as sns
import multiprocessing as mp
import json
import sys
from scipy.stats import hypergeom
from tqdm import tqdm
import networkx.algorithms.community as nx_comm
import NetworkMetrics as metrics
from scipy import stats
import csv
from collections import defaultdict
from matplotlib.colors import ListedColormap, Normalize
import seaborn as sns
import copy
import random
from cmapPy.pandasGEXpress import parse
from scipy.stats import pearsonr
import matplotlib.style
import matplotlib as mpl
from scipy import stats
import bisect
import ast


mpl.style.use('default')
mpl.style.use('classic')



In [ ]:

path_interactome = "./data/PPI_2022.csv"
G = nx.from_pandas_edgelist(pd.read_csv(path_interactome), 'HGNC_Symbol.1', 'HGNC_Symbol.2')

self_loops = [(u, v) for u, v in G.edges() if u == v]
G.remove_edges_from(self_loops)

connected_components = list(nx.connected_components(G))
lcc = max(len(component) for component in connected_components)



interactome2022 - ppi


In [10]:
path = "./data/Gene_hallmarks.csv"
gene_hallmarks_extended = pd.read_csv(path)
all_hallmarks = gene_hallmarks_extended['aging_mechanisms_group'].unique()
gene_hallmarks_extended = gene_hallmarks_extended[gene_hallmarks_extended['confidence'] <= 4]

In [6]:
path = "./CMap_data/level5_beta_trt_cp_n720216x12328.gctx"
gct = parse(path)
gene_signature = gct.data_df
gene_signature_column = list(gene_signature.columns)

In [12]:
path = "./CMap_data/siginfo_beta.txt"
siginfo = pd.read_csv(path,  sep='\t')

/var/folders/7t/jw2g8pt96hn3r4zmn_p719h00000gn/T/ipykernel_42786/619785987.py:2: DtypeWarning: Columns (0,3,4,28,29) have mixed types. Specify dtype option on import or set low_memory=False.
  siginfo = pd.read_csv(path,  sep='\t')


In [16]:
path = "./CMap_data/geneinfo_beta.txt"
geneinfo = pd.read_csv(path,  sep='\t')

In [18]:
path = "./CMap_data/compoundinfo_beta.txt"
compoundinfo = pd.read_csv(path,  sep='\t')
compoundinfo['cmap_name'] = compoundinfo['cmap_name'].str.lower()
compoundinfo['compound_aliases'] = compoundinfo['compound_aliases'].str.lower()
cmap_compounds = list(compoundinfo['cmap_name'].unique())
cmap_compounds_aliases = list(compoundinfo['compound_aliases'].unique())

In [20]:
path = "./data/all_drugbank_drugs.csv"
drugbank = pd.read_csv(path)

In [22]:
aging_mechanisms_groups = ['Exhaustion of stem cells',
              'Altered intercellular communication',
                        'Epigenetic alterations',
                'Mitochondrial dysfunction',
                 'Loss of proteostasis',
                      'Changes in the extracellular matrix structure',
                'Deregulated nutrient sensing',
                    'Genomic instability',
                 'Cell senescence',
                 'Disabled macroautophagy',
                  'Telomere attrition'        ]

In [25]:
amg = 'Exhaustion of stem cells' #Choose hallmark to calculate pAGE

In [27]:
drug_evidence_short = pd.read_csv("./Proximity and pAGE results/drug_evidence_" + amg + ".csv")

In [31]:
path = "./data/age-related-changes.tsv"
age_related_changes = pd.read_csv(path, delimiter = '\t')
age_related_changes = age_related_changes[~(age_related_changes['p value'] == '>0.01')]
age_related_changes = age_related_changes[age_related_changes['model organism'] == 'human']

UDEG = []
DDEG = []
count = 0
for gene in set(age_related_changes['HGNC']):
    A = age_related_changes[age_related_changes['change type'] == 'increased gene expression']
    B = age_related_changes[age_related_changes['change type'] == 'decreased gene expression']
    if len(A[A['HGNC'] == gene]) > 0:
        UDEG.append(gene)
    if len(B[B['HGNC'] == gene]) > 0:
        DDEG.append(gene)


In [ ]:
for level in [1,2,3,4,5]:
    path = "./data/Gene_hallmarks.csv"
    gene_hallmarks_extended = pd.read_csv(path)
    all_hallmarks = gene_hallmarks_extended['aging_mechanisms'].unique()
    gene_hallmarks_extended = gene_hallmarks_extended[gene_hallmarks_extended['confidence'] <= level]
    for idx_amg, amg in enumerate(aging_mechanisms_groups):
        pAGE = []
        pAGE_z = []
        am_genes = set(gene_hallmarks_extended[gene_hallmarks_extended['aging_mechanisms_group'] == amg]['GeneId'])
        am_genes_ids = list(geneinfo[geneinfo['gene_symbol'].isin(am_genes)]['gene_id'].unique())
        if len(am_genes) == 0:
                pAGE = [-2 for i in drug_evidence_short['drug_name']]
                pAGE_z = [0 for i in drug_evidence_short['drug_name']]
                drug_evidence_short['Cmap pAGE ' + amg+ ' level' + str(level)] = pAGE
                drug_evidence_short['Cmap pAGE ' + amg+ ' level' + str(level) + ' z_score'] = pAGE_z
                continue
        
        for idxx, compound in enumerate(drug_evidence_short['drug_name']):
            if not compound in cmap_compounds:
                pAGE.append(-2)
                pAGE_z.append(0)
                continue
                
            


            c_ids = list(compoundinfo[compoundinfo['cmap_name'] == compound.lower()]['pert_id'].unique())
            A = siginfo[(siginfo['pert_mfc_id'].isin(c_ids)) & (siginfo['cell_mfc_name'] == 'MCF7') & (siginfo['pert_dose_unit'] == 'uM')]
            if len(A) == 0:
                pAGE.append(-2)
                pAGE_z.append(0)
                continue
                
            max_dose = max(list(A['pert_dose']))
            A = A[A['pert_dose'] == max_dose]
            max_row = A.loc[A['cc_q75'].idxmax()]

            gctx_file = "./CMap_data/level5_beta_trt_cp_n720216x12328.gctx"


            # Example: Filter columns based on a criterion (e.g., specific column names or metadata)
            desired_columns = [max_row['sig_id']]  # Replace with actual column IDs
            
            # Step 2: Load only the specified columns
            gct_data = parse.parse(gctx_file, cid=desired_columns)
            
            # Access the data_df (only desired columns loaded)
            gene_signature = gct_data.data_df
   



            d_count = 0
            module_count = 0
            for gene in am_genes:
                g_id = list(geneinfo[geneinfo['gene_symbol'] == gene]['gene_id'].unique())
                if len(g_id) == 0:
                    continue
                if gene in DDEG and gene in UDEG:
                    continue
                if (not gene in DDEG) and (not gene in UDEG):
                    continue
                    
                g_zscore = gene_signature.loc[str(g_id[0]),max_row['sig_id']]
                
                    
                if gene in UDEG and g_zscore < 0:
                    d_count = d_count +1
                elif gene in DDEG and g_zscore > 0:
                    d_count = d_count +1
                elif gene in UDEG and g_zscore > 0:
                    d_count = d_count -1
                elif gene in DDEG and g_zscore < 0:
                    d_count = d_count -1
                
                module_count = module_count + 1




            if module_count != 0:
                pAGE_val = d_count / module_count
                pAGE.append(pAGE_val)
                sigma = 1 / np.sqrt(module_count)
                pAGE_z.append(pAGE_val / sigma)
            else:
                pAGE.append(0)
                pAGE_z.append(0)
                
                
        drug_evidence_short['Cmap pAGE ' + amg+ ' level' + str(level)] = pAGE
        drug_evidence_short['Cmap pAGE ' + amg+ ' level' + str(level) + ' z_score'] = pAGE_z
